# RGB Extraction v5 — embedding original dan horizontal flip


## 1. Lokasi dan verifikasi CNN yang telah dilatih


In [1]:
from pathlib import Path
import gc, hashlib, json, os, time
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 30092026
RUN_ID = "cifar10_7fitur_v5"
VARIANT = "rgb"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / VARIANT
MODEL_PATH = OUT / "model" / "cnn.keras"
REPORT_PATH = OUT / "laporan" / "cnn_training.json"
FEATURE_PATH = OUT / "fitur" / "train_features.npz"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


if not MODEL_PATH.is_file() or not REPORT_PATH.is_file():
    raise FileNotFoundError(
        "Jalankan notebook Training " + VARIANT + " v5 terlebih dahulu"
    )
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
if report["cnn_sha256"] != sha256_file(MODEL_PATH) or report["run_id"] != RUN_ID:
    raise ValueError("Checkpoint CNN dan laporan tidak cocok")
print("CNN:", MODEL_PATH, "| SHA:", report["cnn_sha256"][:16])


I0000 00:00:1791576354.711823   71376 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791576354.747812   71376 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1791576355.694242   71376 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


CNN: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/outputs/cifar10_7fitur_v5/rgb/model/cnn.keras | SHA: 82cbe74990f369b5


## 2. Muat official training CIFAR-10 dan split tetap


In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
idx_train, idx_val = train_test_split(
    np.arange(50000), test_size=0.2, stratify=y_all, random_state=SEED
)
idx_train, idx_val = np.sort(idx_train), np.sort(idx_val)
split_hash = hashlib.sha256(idx_train.tobytes() + idx_val.tobytes()).hexdigest()
if split_hash != report["split_hash"] or x_all.shape != (50000, 32, 32, 3):
    raise ValueError("CIFAR-10 atau split tidak sesuai dengan CNN")
print("Split:", split_hash)


Split: 16004a1ac8a89ac92ba6280367a7fb7b284c3643abb1c9d10409d6739ef31305


## 3. Konversi citra identik dengan notebook Training


In [3]:
def prepare_images(x, variant):
    rgb = np.asarray(x, dtype=np.float32) / np.float32(255.0)
    if variant == "rgb":
        return rgb
    if variant == "avg":
        return np.mean(rgb, axis=-1, keepdims=True, dtype=np.float32)
    if variant == "ntsc":
        weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
        return np.sum(rgb * weights, axis=-1, keepdims=True, dtype=np.float32)
    raise ValueError("Varian tidak dikenal: " + variant)


## 4. Muat layer embedding tanpa mengubah bobot


In [4]:
cnn = tf.keras.models.load_model(MODEL_PATH, compile=False)
extractor = tf.keras.Model(cnn.input, cnn.get_layer("embedding").output)
if int(extractor.output.shape[-1]) != 512:
    raise ValueError("Dimensi embedding harus 512")
print("Extractor:", extractor.output.shape)


I0000 00:00:1791576358.398487   71376 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3672 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Extractor: (None, 512)


## 5. Ekstraksi bertahap dalam inference mode


In [5]:
A = np.empty((50000, 512), np.float32)
B = np.empty((50000, 512), np.float32)
began = time.perf_counter()
for begin in range(0, 50000, 128):
    end = min(begin + 128, 50000)
    x = prepare_images(x_all[begin:end], VARIANT)
    original = extractor(x, training=False).numpy().astype(np.float32)
    flipped = (
        extractor(np.ascontiguousarray(x[:, :, ::-1, :]), training=False)
        .numpy()
        .astype(np.float32)
    )
    A[begin:end] = original
    B[begin:end] = (original + flipped) * np.float32(0.5)
    if end % 6400 == 0 or end == 50000:
        print(VARIANT, "fitur", end, "/50000", flush=True)
if not np.isfinite(A).all() or not np.isfinite(B).all():
    raise ValueError("Fitur mengandung NaN/Inf")
print("Ekstraksi selesai, detik:", round(time.perf_counter() - began, 1))
del extractor, cnn
tf.keras.backend.clear_session()
gc.collect()


I0000 00:00:1791576359.426365   71376 cuda_dnn.cc:461] Loaded cuDNN version 92600


rgb fitur 6400 /50000


rgb fitur 12800 /50000


rgb fitur 19200 /50000


rgb fitur 25600 /50000


...

rgb fitur 38400 /50000


rgb fitur 44800 /50000


rgb fitur 50000 /50000


Ekstraksi selesai, detik: 51.5


0

## 6. Simpan arsip fitur serta provenance


In [6]:
if FEATURE_PATH.exists():
    with np.load(FEATURE_PATH, allow_pickle=False) as old:
        if (
            str(old["run_id"]) != RUN_ID
            or str(old["cnn_sha256"]) != report["cnn_sha256"]
            or str(old["split_hash"]) != split_hash
            or old["features_A"].shape != (50000, 512)
            or old["features_B"].shape != (50000, 512)
            or not np.array_equal(old["indices"], np.arange(50000))
            or not np.array_equal(old["y"], y_all)
            or not np.array_equal(old["train_idx"], idx_train)
            or not np.array_equal(old["val_idx"], idx_val)
        ):
            raise ValueError("Arsip fitur lama berbeda; gunakan run baru")
    print("Arsip yang cocok sudah ada:", FEATURE_PATH)
else:
    temporary = FEATURE_PATH.with_suffix(".npz.tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(
            stream,
            features_A=A,
            features_B=B,
            y=y_all,
            indices=np.arange(50000),
            train_idx=idx_train,
            val_idx=idx_val,
            run_id=RUN_ID,
            variant=VARIANT,
            split_hash=split_hash,
            cnn_sha256=report["cnn_sha256"],
            feature_layer="embedding",
            preprocessing_version="v5-float32-div255-original-flip",
            seed=SEED,
        )
    os.replace(temporary, FEATURE_PATH)
print("Arsip:", FEATURE_PATH, "| SHA:", sha256_file(FEATURE_PATH)[:16])


Arsip: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/outputs/cifar10_7fitur_v5/rgb/fitur/train_features.npz | SHA: 0c316adc31fb88fb
